# Crisp on Colab

**Crisp** is a Lisp for writing GPU kernels.  It compiles to PTX (NVIDIA) and SPIR-V (Intel), and it can
**differentiate a kernel**: one `--differentiate` compile produces the forward kernel *and* its gradient kernel.

This notebook installs the compiler with `pip`, compiles kernels **from Python strings at run time**, runs
them on PyTorch tensors, and uses the gradient kernel Crisp writes as a `torch.autograd` backward.

Needs a GPU runtime (*Runtime → Change runtime type → T4 GPU*; the free T4 is plenty).
Source, docs and the language guide: <https://github.com/cperkinscperkins/crisp>

In [ ]:
# 0.5 is a release candidate on TestPyPI for now; once released this is just  pip install crisp-gpu
!pip install -q -i https://test.pypi.org/simple/ crisp-gpu==0.5.0rc1

In [ ]:
import time, torch, crisp_gpu, crisp_gpu.torch as ct
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(), ct.arch(), "| crisp_gpu", crisp_gpu.__version__, "| torch", torch.__version__)

## 1. A kernel is a string

`square_scale` computes `Y[i] = X[i]² · W[i]`.  The declaration gives its signature (two input vectors, one
`&out` vector) and its launch geometry: one thread per element of `X`, 256 threads per block.
`loop-vector-stride` walks the elements this thread owns.

`crisp_gpu.compile` needs no GPU at all: it returns the PTX and a description of every kernel and parameter.

In [ ]:
SRC = """
(def-type vec-t (vector float :address-space :global :align :compact))

(def-kernel square_scale (X W &out Y)
  (declare #'(vec-t vec-t &out vec-t => nil)
           (global-size :derive-from X :strategy :one-thread-per)
           (local-size  :set-to 256))
  (loop-vector-stride X (i)
    (set! (~ Y i) (* (~ X i) (~ X i) (~ W i)))))
"""
t = time.perf_counter()
mod = crisp_gpu.compile(SRC, target="ptx", flags=[f"--ir-target-arch={ct.arch()}"])
print(f"compiled in {(time.perf_counter() - t) * 1e3:.0f} ms")
k = mod.kernel("square_scale")
for p in k.params:
    print(f"  {p.name:3} {p.direction:4} {p.kind} of {p.element_type}, rank {p.rank}")
print("\n".join(mod.binary.decode().splitlines()[:12]))

## 2. Run it on PyTorch tensors

`ct.compile` compiles for the GPU you have and loads the PTX into PyTorch's CUDA context.  `launch` takes
arguments by parameter name; the grid comes from the kernel's own declaration.

In [ ]:
m = ct.compile(SRC)
x = torch.randn(1 << 20, device="cuda"); w = torch.randn_like(x); y = torch.empty_like(x)
grid, block = m.launch("square_scale", x=x, w=w, y=y)
torch.cuda.synchronize()
print("grid", grid, "block", block)
print("max |crisp - torch| =", (y - x * x * w).abs().max().item())

## 3. The gradient comes from the compiler

With `--differentiate` the module holds a second kernel, `square_scale`'s gradient, whose parameters say
which adjoint each one carries.  `ct.autograd_function` makes the pair a `torch.autograd.Function`: the
forward runs Crisp's kernel, and `backward()` runs the gradient kernel Crisp derived.  No hand-written backward.

In [ ]:
mad = ct.compile(SRC, flags=["--differentiate"])
for kk in mad.module.kernels:
    print(kk.name, "(gradient of " + kk.grad_of + ")" if kk.grad_of else "",
          [(p.name, p.direction, p.adjoint_of) for p in kk.params if p.purpose == "declared"])

f = ct.autograd_function(mad, "square_scale")
x = torch.randn(1 << 20, device="cuda", requires_grad=True); w = torch.randn_like(x, requires_grad=True)
f(x, w).sum().backward()
gx, gw = x.grad.clone(), w.grad.clone()
x.grad = w.grad = None
(x * x * w).sum().backward()                      # torch's own autograd, for comparison
print("max dx error", (gx - x.grad).abs().max().item(), "  max dw error", (gw - w.grad).abs().max().item())

## 4. Compile at run time: specialize to values you only know now

Because compiling is a function call, a kernel can be written *for this run*: here the coefficients of
`y = a·x³ + b·x` are baked in as constants, and each specialization gets its own derived gradient.

In [ ]:
def poly_kernel(a, b):
    """y = a*x^3 + b*x, with a and b compiled in."""
    return f"""
(def-type vec-t (vector float :address-space :global :align :compact))

(def-kernel poly (X &out Y)
  (declare #'(vec-t &out vec-t => nil)
           (global-size :derive-from X :strategy :one-thread-per)
           (local-size  :set-to 256))
  (loop-vector-stride X (i)
    (let ((xi (~ X i)))
      (set! (~ Y i) (+ (* {a:.6f} xi xi xi) (* {b:.6f} xi))))))
"""

for a, b in [(1.5, -0.25), (0.1, 2.0), (-3.0, 0.5)]:
    t = time.perf_counter()
    f = ct.autograd_function(ct.compile(poly_kernel(a, b), flags=["--differentiate"]), "poly")
    ms = (time.perf_counter() - t) * 1e3
    x = torch.linspace(-2, 2, 1001, device="cuda", requires_grad=True)
    y = f(x); y.sum().backward()
    xd = x.detach()
    print(f"a={a:5} b={b:5}  compiled+loaded in {ms:4.0f} ms   "
          f"y error {(y.detach() - (a * xd**3 + b * xd)).abs().max().item():.1e}   "
          f"dy/dx error {(x.grad - (3 * a * xd**2 + b)).abs().max().item():.1e}")

## 5. Timing

A memory-bound kernel, timed with CUDA events.  PyTorch eager runs this as two kernels (`x*x`, then `*w`);
`torch.compile` would fuse them too.  The point is not the race but that the Crisp kernel runs at the
GPU's memory bandwidth (a T4 peaks at 320 GB/s).

In [ ]:
def bench(fn, n=100):
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    fn(); torch.cuda.synchronize(); s.record()
    for _ in range(n):
        fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / n * 1e3          # microseconds

x = torch.randn(1 << 24, device="cuda"); w = torch.randn_like(x); y = torch.empty_like(x)
t_crisp = bench(lambda: m.launch("square_scale", x=x, w=w, y=y))
t_torch = bench(lambda: torch.mul(x * x, w, out=y))
gb = 3 * x.numel() * 4 / 1e9
print(f"crisp       {t_crisp:6.0f} us   {gb / (t_crisp * 1e-6):4.0f} GB/s")
print(f"torch eager {t_torch:6.0f} us")

## 6. Standalone host code

The `crisp-compile` command (installed by pip) can also *hoist* a kernel: write a C++ program that loads
and launches it through the CUDA driver API, with the buffers its declaration describes.  Colab has `nvcc`,
so we can build and run one.  It fills `x` and `w` with 0 1 2 3, so `y` should be 0 1 8 27.

In [ ]:
%%bash
cat > sq.crisp <<'EOF'
(def-type vec-t (vector float :address-space :global :align :compact))

(def-kernel square_scale (X W &out Y)
  (declare #'(vec-t vec-t &out vec-t => nil)
           (global-size :derive-from X :strategy :one-thread-per)
           (local-size  :set-to 256))
  (loop-vector-stride X (i)
    (set! (~ Y i) (* (~ X i) (~ X i) (~ W i)))))
EOF
ARCH=$(python -c "import torch; a, b = torch.cuda.get_device_capability(); print(f'sm_{a}{b}')")
crisp-compile --ir-target=ptx --ir-target-arch=$ARCH --hoist=cuda sq.crisp
nvcc -o sq_host sq_square_scale_CUDA.cu -lcuda && ./sq_host

## Where next

* The language guide: [writing-crisp.md](https://github.com/cperkinscperkins/crisp/blob/main/docs/writing-crisp.md)
  (idioms, the rules that trip people up, and exactly which builtins exist).
* The design document: <https://cperkinscperkins.github.io/crisp/docs/index.html>
* Intel GPUs: `crisp_gpu.compile(SRC, target="spv")` gives SPIR-V, and `--hoist=l0` a Level Zero host program.